# Проект спринта 27. Часть 2: эксперименты и обучение

Это вторая тетрадка проекта по извлечению именованных сущностей (NER).
Чтобы приступить к работе с ней, вы должны завершить работу с первой тетрадкой, `ydp_27p_eda.ipynb`, в которой вы исследовали данные и изучили логику токенизации и выравнивания меток.

В этой части проекта вы настроите оценку качества моделей в целом и на уровне целых сущностей, получите baseline, проведёте два эксперимента с разным объёмом обучающих данных, сравните модели, проанализируете ошибки и проверите итоговую модель на собственных текстах.

**Модель:** `bert-base-multilingual-cased`.

## Подготовка окружения и обеспечение воспроизводимости

Установите зависимости, импортируйте библиотеки, зафиксируйте `SEED` и проверьте
доступное вычислительное устройство.

In [1]:
# Установка библиотек
%pip install -q numpy pandas matplotlib torch transformers datasets accelerate evaluate seqeval spacy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00


In [2]:
from collections import Counter
from importlib.metadata import version

import evaluate
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from seqeval.metrics.sequence_labeling import get_entities
from spacy import displacy
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
    pipeline,
    set_seed,
)


In [3]:
SEED = 42
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {device}")
print("Версии: " + ", ".join(
    f"{package}={version(package)}"
    for package in [
        "torch",
        "transformers",
        "datasets",
        "accelerate",
        "evaluate",
        "seqeval",
        "spacy",
    ]
))


Используемое устройство: cuda
Версии: torch=2.11.0+cu128, transformers=5.15.1, datasets=4.0.0, accelerate=1.14.0, evaluate=0.4.6, seqeval=1.2.2, spacy=3.8.16


## Подготовка данных для BERT

Этот блок повторяет подготовку из первой тетрадки: загружает WikiANN, собирает словари
меток, токенизирует предложения и выравнивает NER-метки с подтокенами BERT. Код приведён целиком — дополнять его не нужно.

Напомним политику выравнивания: первый токен слова получает исходную метку,
остальные токены, специальные токены и паддинг получают `-100` и исключаются
из расчёта функции потерь.

In [4]:
# Загрузка русскоязычной конфигурации WikiANN и словарей меток
raw_datasets = load_dataset("unimelb-nlp/wikiann", "ru")

label_list = raw_datasets["train"].features["ner_tags"].feature.names
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}
ENTITY_TYPES = sorted({
    label.split("-")[-1]
    for label in label_list
    if label != "O"
})

print(f"Метки: {label_list}")
print(f"Типы сущностей: {ENTITY_TYPES}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/158k [00:00<?, ?B/s]

ru/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  809kB            

ru/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

ru/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  816kB            

...

ru/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Метки: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']
Типы сущностей: ['LOC', 'ORG', 'PER']


In [5]:
model_checkpoint = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

In [6]:
def tokenize_and_align_labels(examples):
    """Токенизирует слова и выравнивает NER-метки с подтокенами."""
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [7]:
tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)

# Эти переменные понадобятся для работы моделей
train_full = tokenized_datasets["train"]
eval_dataset = tokenized_datasets["validation"]
test_dataset = tokenized_datasets["test"]

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# Контрольная проверка: длины полей одного примера должны совпадать
example = train_full[0]
assert (
    len(example["input_ids"])
    == len(example["attention_mask"])
    == len(example["labels"])
)

print(f"Обучающая выборка:   {len(train_full)}")
print(f"Валидационная:       {len(eval_dataset)}")
print(f"Тестовая:            {len(test_dataset)}")

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Обучающая выборка:   20000
Валидационная:       10000
Тестовая:            10000


## Подготовка к обучению

Ниже представлены несколько блоков кода, которые понадобятся вам для работы с моделями:

- код, который фиксирует общие параметры экспериментов;
- функция для инициализации свежей модели;
- функция для обучения модели.

Работая с моделями, используйте валидационный срез для выбора `learning_rate`, а полную тестовую выборку — только для итоговой оценки моделей. Между экспериментами меняется размер обучающей выборки, остальные настройки остаются одинаковыми.


In [8]:
# Настройки моделей
FEWSHOT_SIZE = 100
LARGE_SIZE = 1000
NUM_EPOCHS = 5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 64
EVAL_SUBSET_SIZE = 1000
CANDIDATE_LEARNING_RATES = (5e-5, 2e-5)

eval_small = eval_dataset.shuffle(seed=SEED).select(range(EVAL_SUBSET_SIZE))
print(
    f"Валидация во время обучения: "
    f"{len(eval_small)} из {len(eval_dataset)} примеров"
)


Валидация во время обучения: 1000 из 10000 примеров


In [9]:
def init_model():
    """Создаёт свежую модель token classification с одинаковой инициализацией."""
    set_seed(SEED)
    return AutoModelForTokenClassification.from_pretrained(
        model_checkpoint,
        num_labels=len(label_list),
        id2label=id2label,
        label2id=label2id,
    )


In [10]:
def train_model(train_dataset, lr, output_dir):
    """Создаёт свежую модель и обучает её с заданным learning rate."""
    model = init_model()
    steps_per_epoch = int(np.ceil(len(train_dataset) / TRAIN_BATCH_SIZE))

    args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        # Сохраняем чекпойнт каждой эпохи и в конце берём лучший по валидационному F1
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        learning_rate=lr,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=0.01,
        logging_steps=max(steps_per_epoch // 2, 1),
        seed=SEED,
        report_to="none",
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_small,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    return trainer


all_results = {}

## Настройка метрик

Реализуйте преобразование логитов и меток в строковые IOB2-теги. Исключите позиции
`-100`. `compute_metrics` должен возвращать общие `precision`, `recall`, `f1`,
`accuracy`, а также `precision`, `recall`, `f1` по каждому типу сущности. Используйте
ключ `f1` для общего `f1` и ключи вида `PER_f1` для отдельных типов меток. Если тип
отсутствует в результате `seqeval`, верните для него `0.0`.


In [11]:
metric = evaluate.load("seqeval")

def align_predictions(predictions, labels):
    """Преобразует логиты и id-метки в строки, исключая позиции -100."""
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for p, l in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for l in label if l != -100]
        for label in labels
    ]
    return true_predictions, true_labels


def compute_metrics(p):
    """Возвращает общие метрики и метрики по PER, ORG, LOC."""
    predictions, labels = p
    true_predictions, true_labels = align_predictions(predictions, labels)
    results = metric.compute(predictions=true_predictions, references=true_labels)
    
    metrics = {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }
    for entity_type in ENTITY_TYPES:
        entity_result = results.get(entity_type, {})
        metrics[f"{entity_type}_precision"] = entity_result.get("precision", 0.0)
        metrics[f"{entity_type}_recall"] = entity_result.get("recall", 0.0)
        metrics[f"{entity_type}_f1"] = entity_result.get("f1", 0.0)

    return metrics

In [12]:
def evaluate_model(trainer, dataset, name):
    """Выполняет один predict и возвращает метрики и строковые теги."""
    output = trainer.predict(dataset)
    true_predictions, true_labels = align_predictions(
        output.predictions,
        output.label_ids,
    )
    results = metric.compute(
        predictions=true_predictions,
        references=true_labels,
    )

    print(f"\n=== {name} ===")
    print(
        f"{'Overall':8} | "
        f"P: {results['overall_precision']:.4f} | "
        f"R: {results['overall_recall']:.4f} | "
        f"F1: {results['overall_f1']:.4f}"
    )
    for entity in ENTITY_TYPES:
        if entity in results:
            print(
                f"{entity:8} | "
                f"P: {results[entity]['precision']:.4f} | "
                f"R: {results[entity]['recall']:.4f} | "
                f"F1: {results[entity]['f1']:.4f}"
            )
        else:
            print(f"{entity:8} | не предсказан ни разу")

    return results, true_predictions, true_labels


## Шаг 4. Создание базовой модели

Baseline — предобученный BERT с новой необученной NER-головой. Не вызывайте
`train()` — сразу оцените свежую модель на полной тестовой выборке.


In [13]:
baseline_model = init_model()
baseline_trainer = Trainer(
    model=baseline_model,
    args=TrainingArguments(
        "bert-ner-baseline",
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        report_to="none",
    ),
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

baseline_results, _, _ = evaluate_model(
    baseline_trainer,
    test_dataset,
    "Baseline без обучения",
)
all_results["baseline"] = baseline_results

del baseline_trainer, baseline_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params


=== Baseline без обучения ===
Overall  | P: 0.0218 | R: 0.0781 | F1: 0.0341
LOC      | P: 0.0574 | R: 0.1452 | F1: 0.0823
ORG      | P: 0.0234 | R: 0.0663 | F1: 0.0346
PER      | P: 0.0009 | R: 0.0054 | F1: 0.0016


### Выводы о базовой модели

Baseline без обучения даёт F1 = 0.034 - голова классификации инициализирована случайно, поэтому предсказания фактически случайны, а перекос в сторону LOC объясняется лишь частотностью этого класса в данных. Это нижняя граница, относительно которой будет оцениваться прирост от дообучения.


## Шаг 5. Эксперимент на малой выборке

Выберите 100 воспроизводимых случайных примеров из полной обучающей выборки. Обучите две свежие
модели с `learning_rate=5e-5` и `learning_rate=2e-5`. Выберите лучший вариант по
F1 на одном и том же валидационном срезе.


In [14]:
fewshot_train = train_full.shuffle(seed=SEED).select(range(FEWSHOT_SIZE))

In [15]:
lr_a = CANDIDATE_LEARNING_RATES[0]
trainer_lr_a = train_model(fewshot_train, lr_a, output_dir=f"./fewshot_lr_{lr_a}")
val_f1_a = trainer_lr_a.evaluate()["eval_f1"]
print(f"Вариант A: lr={lr_a:.0e} | val F1: {val_f1_a:.4f}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy,Loc Precision,Loc Recall,Loc F1,Org Precision,Org Recall,Org F1,Per Precision,Per Recall,Per F1
1,1.297129,1.069879,0.350263,0.161943,0.221484,0.646113,0.000000,0.000000,0.000000,0.114754,0.036745,0.055666,0.416107,0.479381,0.445509
2,0.825549,0.778612,0.502618,0.310931,0.384192,0.736539,0.329730,0.130901,0.187404,0.257732,0.131234,0.173913,0.709091,0.703608,0.706339
3,0.504608,0.629151,0.463803,0.472065,0.467897,0.800266,0.391892,0.373391,0.382418,0.357333,0.351706,0.354497,0.627854,0.708763,0.665860
4,0.310048,0.574861,0.455189,0.468826,0.461907,0.806461,0.386609,0.384120,0.385361,0.366366,0.320210,0.341737,0.584034,0.716495,0.643519
5,0.290548,0.555620,0.464477,0.492308,0.477987,0.814132,0.390782,0.418455,0.404145,0.383523,0.354331,0.368349,0.606987,0.716495,0.657210


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy,Loc Precision,Loc Recall,Loc F1,Org Precision,Org Recall,Org F1,Per Precision,Per Recall,Per F1
0.290548,0.555620,5,0.464477,0.492308,0.477987,0.814132,0.390782,0.418455,0.404145,0.383523,0.354331,0.368349,0.606987,0.716495,0.657210


Вариант A: lr=5e-05 | val F1: 0.4780


In [16]:
lr_b = CANDIDATE_LEARNING_RATES[1]
trainer_lr_b = train_model(fewshot_train, lr_b, output_dir=f"./fewshot_lr_{lr_b}")
val_f1_b = trainer_lr_b.evaluate()["eval_f1"]
print(f"Вариант B: lr={lr_b:.0e} | val F1: {val_f1_b:.4f}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy,Loc Precision,Loc Recall,Loc F1,Org Precision,Org Recall,Org F1,Per Precision,Per Recall,Per F1
1,1.400363,1.346363,0.000000,0.000000,0.000000,0.543886,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,1.171205,1.097070,0.531722,0.142510,0.224777,0.640360,0.100000,0.002146,0.004202,0.040000,0.002625,0.004926,0.587838,0.448454,0.508772
3,0.945758,0.966092,0.451839,0.208907,0.285714,0.677239,0.244444,0.023605,0.043053,0.123529,0.055118,0.076225,0.634831,0.582474,0.607527
4,0.685690,0.890454,0.521173,0.259109,0.346133,0.702021,0.375000,0.051502,0.090566,0.143564,0.076115,0.099485,0.767241,0.688144,0.725543
5,0.713853,0.864801,0.526814,0.270445,0.357410,0.709397,0.329412,0.060086,0.101633,0.170854,0.089239,0.117241,0.777143,0.701031,0.737127


/usr/local/lib/python3.13/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.13/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy,Loc Precision,Loc Recall,Loc F1,Org Precision,Org Recall,Org F1,Per Precision,Per Recall,Per F1
0.713853,0.864801,5,0.526814,0.270445,0.357410,0.709397,0.329412,0.060086,0.101633,0.170854,0.089239,0.117241,0.777143,0.701031,0.737127


Вариант B: lr=2e-05 | val F1: 0.3574


In [17]:
if val_f1_a >= val_f1_b:
    best_lr, trainer_fewshot, trainer_to_drop = lr_a, trainer_lr_a, trainer_lr_b
else:
    best_lr, trainer_fewshot, trainer_to_drop = lr_b, trainer_lr_b, trainer_lr_a
print(f"\nЛучший learning_rate: {best_lr:.0e}")

fewshot_results, pred_tags_fewshot, true_tags_test = evaluate_model(
    trainer_fewshot, test_dataset, f"Few-shot ({FEWSHOT_SIZE}), lr={best_lr:.0e}"
)
all_results["fewshot"] = fewshot_results

# Освободите модели, которые больше не нужны
del trainer_to_drop
torch.cuda.empty_cache()


Лучший learning_rate: 5e-05



=== Few-shot (100), lr=5e-05 ===
Overall  | P: 0.4398 | R: 0.4872 | F1: 0.4623
LOC      | P: 0.3826 | R: 0.4357 | F1: 0.4075
ORG      | P: 0.3808 | R: 0.3557 | F1: 0.3678
PER      | P: 0.5558 | R: 0.7048 | F1: 0.6215


### Выводы о модели, дообученной на малой выборке

Валидационный F1: 0.4780 при lr=5e-5 против 0.3574 при lr=2e-5 — выбран вариант A, так как при меньшем learning rate модель за 5 эпох просто не успела обучиться. Хуже всего на тесте распознаётся ORG (F1 = 0.3678): организации разнообразнее по форме, чем имена, и 100 примеров для них мало.

## Шаг 6. Эксперимент на увеличенной выборке

Выберите 1000 воспроизводимых случайных примеров из полной обучающей выборки.

Инициализируйте новую модель, используйте лучший `learning_rate` и не меняйте остальные настройки.


In [18]:
large_train = train_full.shuffle(seed=SEED).select(range(LARGE_SIZE))

trainer_best_lr = train_model(large_train, best_lr, output_dir=f"./large_lr_{best_lr}")
large_results, pred_tags_large, true_tags_large = evaluate_model(
    trainer_best_lr, test_dataset, f"Large ({LARGE_SIZE}), lr={best_lr:.0e}"
)

assert true_tags_large == true_tags_test, "Истинные теги тестовой выборки различаются"
print("Истинные теги совпадают")

all_results["large"] = large_results

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy,Loc Precision,Loc Recall,Loc F1,Org Precision,Org Recall,Org F1,Per Precision,Per Recall,Per F1
1,0.366653,0.321776,0.699695,0.743320,0.720848,0.908541,0.721591,0.817597,0.766600,0.581006,0.545932,0.562923,0.772300,0.847938,0.808354
2,0.263042,0.280464,0.724926,0.795951,0.758780,0.921375,0.728597,0.858369,0.788177,0.568720,0.629921,0.597758,0.890909,0.884021,0.887451
3,0.126662,0.291412,0.777778,0.816194,0.796523,0.925800,0.818777,0.804721,0.811688,0.648889,0.766404,0.702768,0.878866,0.878866,0.878866
4,0.047172,0.320597,0.782542,0.827530,0.804408,0.929636,0.794872,0.864807,0.828366,0.660050,0.698163,0.678571,0.891414,0.909794,0.900510
5,0.042590,0.334965,0.792570,0.829150,0.810447,0.927423,0.835821,0.841202,0.838503,0.655012,0.737533,0.693827,0.890863,0.904639,0.897698


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte


=== Large (1000), lr=5e-05 ===
Overall  | P: 0.7845 | R: 0.8313 | F1: 0.8072
LOC      | P: 0.8091 | R: 0.8419 | F1: 0.8251
ORG      | P: 0.6635 | R: 0.7386 | F1: 0.6990
PER      | P: 0.9037 | R: 0.9244 | F1: 0.9139
Истинные теги совпадают


## Шаг 7. Сравнение результатов

Соберите в одной таблице данные по baseline и моделям, обученным на 100 и 1000 примерах.

Покажите общие `precision`, `recall`, `f1` и `f1` для `PER`, `ORG`, `LOC`.


In [19]:
def results_to_row(results):
    row = {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }
    for entity_type in ENTITY_TYPES:
        row[f"{entity_type}_f1"] = results.get(entity_type, {}).get("f1", 0.0)
    return row

comparison_df = pd.DataFrame(
    {name: results_to_row(res) for name, res in all_results.items()}
).T
print(comparison_df.round(4))

          precision  recall      f1  LOC_f1  ORG_f1  PER_f1
baseline     0.0218  0.0781  0.0341  0.0823  0.0346  0.0016
fewshot      0.4398  0.4872  0.4623  0.4075  0.3678  0.6215
large        0.7845  0.8313  0.8072  0.8251  0.6990  0.9139


In [20]:
absolute = comparison_df.loc["large"] - comparison_df.loc["fewshot"]
relative = absolute / comparison_df.loc["fewshot"] * 100

growth_df = pd.DataFrame({
    "few-shot": comparison_df.loc["fewshot"],
    "large": comparison_df.loc["large"],
    "абс.": absolute,
    "отн., %": relative,
})
print(growth_df.round(4))

entity_gains = {t: absolute[f"{t}_f1"] for t in ENTITY_TYPES}
best_entity = max(entity_gains, key=entity_gains.get)
print(f"\nНаибольший прирост F1: {best_entity} (+{entity_gains[best_entity]:.4f})")

           few-shot   large    абс.   отн., %
precision    0.4398  0.7845  0.3447   78.3835
recall       0.4872  0.8313  0.3441   70.6219
f1           0.4623  0.8072  0.3449   74.6152
LOC_f1       0.4075  0.8251  0.4177  102.5080
ORG_f1       0.3678  0.6990  0.3312   90.0519
PER_f1       0.6215  0.9139  0.2925   47.0602

Наибольший прирост F1: LOC (+0.4177)


### Выводы о результатах сравнения моделей

Baseline давал F1 = 0.0341 (случайные предсказания), few-shot - 0.4623, large - 0.8072. При переходе от 100 к 1000 примерам общий F1 вырос на 0.3449 (+74.6%), precision и recall выросли почти одинаково: +0.3447 (+78.4%) и +0.3441 (+70.6%).

Наибольший прирост у LOC (+0.4177, качество удвоилось). PER вырос меньше всех (+0.2925), но лишь потому, что был лучшим изначально и остаётся сильнейшим (0.9139), а ORG даже после роста остаётся худшим типом (0.6990).

## Шаг 8. Анализ ошибок

Проанализируйте ошибки на уровне целых сущностей. Используйте сохранённые предсказания, чтобы не запускать повторный `predict()` на полной тестовой выборке.

При анализе ориентируйтесь на типы ошибок из таблицы:

| Тип | Описание |
|---|---|
|Пропуск сущности|Сущность есть в эталоне, но отсутствует в предсказании.|
|Неверный тип|Границы совпали, но тип сущности отличается.|
|Неправильные границы|Предсказание пересекается с эталоном, но границы отличаются.|
|Лишняя сущность|Модель нашла сущность там, где её нет в эталоне.|


In [21]:
test_tokens = raw_datasets["test"]["tokens"]

MISSED = "пропуск сущности"
WRONG_TYPE = "неверный тип"
WRONG_BOUNDARIES = "неправильные границы"
SPURIOUS = "лишняя сущность"
ERROR_KINDS = [MISSED, WRONG_TYPE, WRONG_BOUNDARIES, SPURIOUS]


def classify_errors(true_tags, pred_tags):
    """Сопоставляет сущности и классифицирует расхождения."""
    unmatched_pred = list(get_entities(pred_tags))
    errors = []

    for gold in get_entities(true_tags):
        gold_type, gold_start, gold_end = gold

        exact = next(
            (
                pred
                for pred in unmatched_pred
                if (pred[1], pred[2]) == (gold_start, gold_end)
            ),
            None,
        )
        if exact is not None:
            unmatched_pred.remove(exact)
            if exact[0] != gold_type:
                errors.append({
                    "kind": WRONG_TYPE,
                    "gold": gold,
                    "pred": exact,
                })
            continue

        overlapping = next(
            (
                pred
                for pred in unmatched_pred
                if pred[1] <= gold_end and gold_start <= pred[2]
            ),
            None,
        )
        if overlapping is not None:
            unmatched_pred.remove(overlapping)
            errors.append({
                "kind": WRONG_BOUNDARIES,
                "gold": gold,
                "pred": overlapping,
            })
            continue

        errors.append({"kind": MISSED, "gold": gold, "pred": None})

    for spurious in unmatched_pred:
        errors.append({
            "kind": SPURIOUS,
            "gold": None,
            "pred": spurious,
        })

    return errors


def format_entity(entity, tokens):
    """Возвращает сущность в виде «текст» [ТИП]."""
    if entity is None:
        return "—"
    entity_type, start, end = entity
    return f"«{' '.join(tokens[start:end + 1])}» [{entity_type}]"


### Поиск примеров ошибок

Выведите не менее пяти предложений, в которых ошибается модель, обученная на 1000 примеров. Для каждого покажите текст, эталонные и предсказанные теги, тип ошибки и сущности.


In [22]:
NUM_ERROR_EXAMPLES = 5

shown = 0
for idx, (true_tags, pred_tags) in enumerate(zip(true_tags_test, pred_tags_large)):
    errors = classify_errors(true_tags, pred_tags)
    if not errors:
        continue

    tokens = test_tokens[idx]
    print(f"[{idx}] {' '.join(tokens)}")
    print(f"  эталон:  {true_tags}")
    print(f"  предикт: {pred_tags}")
    for e in errors:
        print(f"  {e['kind']}: {format_entity(e['gold'], tokens)} -> {format_entity(e['pred'], tokens)}")
    print()

    shown += 1
    if shown == NUM_ERROR_EXAMPLES:
        break

[0] ' '' Крус Асуль '' '
  эталон:  ['O', 'O', 'B-ORG', 'I-ORG', 'O', 'O']
  предикт: ['O', 'O', 'B-PER', 'I-PER', 'O', 'O']
  неверный тип: «Крус Асуль» [ORG] -> «Крус Асуль» [PER]

[1] == Награда Республики Гвинея ==
  эталон:  ['O', 'O', 'B-LOC', 'I-LOC', 'O']
  предикт: ['O', 'B-ORG', 'I-ORG', 'I-ORG', 'O']
  неправильные границы: «Республики Гвинея» [LOC] -> «Награда Республики Гвинея» [ORG]

[9] Каррингтон , Хиуорд
  эталон:  ['B-PER', 'I-PER', 'I-PER']
  предикт: ['B-PER', 'I-PER', 'I-ORG']
  неправильные границы: «Каррингтон , Хиуорд» [PER] -> «Каррингтон ,» [PER]
  лишняя сущность: — -> «Хиуорд» [ORG]

[12] 1975 — 1998 — ''не участвовали ''
  эталон:  ['B-ORG', 'O', 'B-ORG', 'O', 'O', 'O', 'O']
  предикт: ['B-LOC', 'O', 'O', 'O', 'O', 'O', 'O']
  неверный тип: «1975» [ORG] -> «1975» [LOC]
  пропуск сущности: «1998» [ORG] -> —

[15] Находится в составе крупной городской агломерации Большое Минью .
  эталон:  ['O', 'O', 'O', 'B-ORG', 'I-ORG', 'I-ORG', 'B-ORG', 'I-ORG', 'O']
  пр

### Сравнение частот ошибок

Посчитайте типы ошибок по всей тестовой выборке для обеих обученных моделей.
Сравните абсолютные значения и доли, затем определите самый частый тип ошибки.


In [23]:
def count_error_kinds(true_tags_list, pred_tags_list):
    """Считает частоты типов ошибок по всему набору."""
    counter = Counter()
    for true_tags, pred_tags in zip(true_tags_list, pred_tags_list):
        for error in classify_errors(true_tags, pred_tags):
            counter[error["kind"]] += 1
    return counter

errors_fewshot = count_error_kinds(true_tags_test, pred_tags_fewshot)
errors_large = count_error_kinds(true_tags_test, pred_tags_large)

error_df = pd.DataFrame({
    "few-shot": pd.Series(errors_fewshot),
    "large": pd.Series(errors_large),
}).fillna(0).astype(int)

error_df["few-shot, %"] = error_df["few-shot"] / error_df["few-shot"].sum() * 100
error_df["large, %"] = error_df["large"] / error_df["large"].sum() * 100
error_df = error_df.sort_values("large", ascending=False)

print(error_df.round(2))

top_error = error_df["large"].idxmax()
print(f"\nСамый частый тип ошибки (large): {top_error} "
      f"({error_df.loc[top_error, 'large']} шт., {error_df.loc[top_error, 'large, %']:.1f}%)")

                      few-shot  large  few-shot, %  large, %
лишняя сущность           3160   1095        33.60     34.77
неправильные границы      3537    939        37.61     29.82
неверный тип               861    747         9.16     23.72
пропуск сущности          1846    368        19.63     11.69

Самый частый тип ошибки (large): лишняя сущность (1095 шт., 34.8%)


### Поиск расхождений между моделями

Найдите примеры, в которых каждая из обученных моделей показывает себя лучше другой. Для этого продемонстрируйте случаи, где модель на малой выборке права, а модель на увеличенной ошибается, и наоборот. Сравнивайте извлечённые сущности, а не только последовательности тегов.


In [24]:
def same_entities(tags_a, tags_b):
    return get_entities(tags_a) == get_entities(tags_b)


def find_disagreements(true_tags_list, tags_a, tags_b, max_examples=2):
    """Возвращает индексы примеров, где одна модель права, а другая ошибается."""
    a_better = []
    b_better = []

    for i, true_tags in enumerate(true_tags_list):
        a_ok = same_entities(tags_a[i], true_tags)
        b_ok = same_entities(tags_b[i], true_tags)

        if a_ok and not b_ok and len(a_better) < max_examples:
            a_better.append(i)
        if b_ok and not a_ok and len(b_better) < max_examples:
            b_better.append(i)
        if len(a_better) >= max_examples and len(b_better) >= max_examples:
            break

    return a_better, b_better

a_better, b_better = find_disagreements(true_tags_test, pred_tags_fewshot, pred_tags_large)

def show_disagreement(idx):
    tokens = raw_datasets["test"][idx]["tokens"]
    print(f"[{idx}] {' '.join(tokens)}")
    print(f"эталон: {get_entities(true_tags_test[idx])}")
    print(f"few-shot: {get_entities(pred_tags_fewshot[idx])}")
    print(f"large: {get_entities(pred_tags_large[idx])}")
    print()


print("Few-shot прав, large ошибается")
for idx in a_better:
    show_disagreement(idx)

print("Large прав, few-shot ошибается")
for idx in b_better:
    show_disagreement(idx)

Few-shot прав, large ошибается
[0] ' '' Крус Асуль '' '
эталон: [('ORG', 2, 3)]
few-shot: [('ORG', 2, 3)]
large: [('PER', 2, 3)]

[9] Каррингтон , Хиуорд
эталон: [('PER', 0, 2)]
few-shot: [('PER', 0, 2)]
large: [('PER', 0, 1), ('ORG', 2, 2)]

Large прав, few-shot ошибается
[2] К тому же сериал ITV Мистер Селфридж был более популярен .
эталон: [('ORG', 4, 4), ('ORG', 5, 6)]
few-shot: [('ORG', 5, 5), ('LOC', 6, 6)]
large: [('ORG', 4, 4), ('ORG', 5, 6)]

[4] Паро ( река )
эталон: [('LOC', 0, 3)]
few-shot: [('PER', 0, 0), ('ORG', 1, 2)]
large: [('LOC', 0, 3)]



### Выводы из анализа ошибок

Large-модель ошибается втрое реже few-shot (3149 против 9404), причём границы и пропуски улучшились сильно, а путаница в типах почти нет - её доля выросла до 23.7%; самая частая ошибка теперь "лишняя сущность" (34.8%). Модель путает ORG с PER в названиях вроде "Крус Асуль" и сбивается на границах у скобок и запятых, но часть таких ошибок - шум WikiANN, где эталон строится по гиперссылкам и пропускает неразмеченные упоминания.


## Шаг 9. Сохранение модели и проверка на новых текстах

Используйте итоговую модель из эксперимента на увеличенной выборке. Получите
предсказания на небольшой фиксированной части тестовой выборки, сохраните модель
и токенизатор, загрузите их обратно и повторите предсказание на тех же примерах.
Результаты до и после загрузки должны полностью совпасть.


In [25]:
save_dir = "./final_model"
test_small = test_dataset.select(range(100))

preds_before = trainer_best_lr.predict(test_small).predictions

trainer_best_lr.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

trainer_best_lr.model = AutoModelForTokenClassification.from_pretrained(save_dir).to(
    trainer_best_lr.model.device
)
preds_after = trainer_best_lr.predict(test_small).predictions

print(f"Логиты совпадают: {np.array_equal(preds_before, preds_after)}")
print(f"Предсказания совпадают: {np.array_equal(np.argmax(preds_before, axis=2), np.argmax(preds_after, axis=2))}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Логиты совпадают: True
Предсказания совпадают: True


### Проверка модели на собственных текстах

Создайте 3–5 предложений, которых нет в WikiANN. Желательно включить однословные
и многословные сущности `PER`, `ORG`, `LOC`, а также сложные или неоднозначные примеры.


In [26]:
ner_pipeline = pipeline(
    "token-classification",
    model=trainer_best_lr.model,
    tokenizer=tokenizer,
    aggregation_strategy="first",
    device=0 if torch.cuda.is_available() else -1,
)

def visualize_predictions(text, pipeline_output):
    """Визуализирует предсказанные сущности через displacy."""
    ents = [
        {
            "start": entity["start"],
            "end": entity["end"],
            "label": entity["entity_group"],
        }
        for entity in pipeline_output
    ]
    displacy.render(
        {"text": text, "ents": ents, "title": None},
        style="ent",
        manual=True,
        jupyter=True,
    )

In [27]:
custom_sentences = [
    "Сергей Брин основал Google в Калифорнии",
    "Штаб-квартира Сбера находится в Москве",
    "Барселона обыграла Реал Мадрид на Камп Ноу",
]

results = ner_pipeline.predict(custom_sentences)
for text, result in zip(custom_sentences, results):
    visualize_predictions(text, result)

## Шаг 10. Финальные выводы

Итоги. Выбран lr=5e-5 (F1 0.4780 против 0.3574 у 2e-5, который за 5 эпох не успел обучиться). Baseline давал 0.0341, few-shot на 100 примерах - 0.4623, на 1000 примерах - 0.8072. Лучше всех PER (0.9139), хуже всех ORG (0.6990), сильнее всего вырос LOC. Ошибок стало втрое меньше, но путаница в типах почти не ушла; самая частая ошибка - лишняя сущность (34.8%). Модель корректно сохраняется и загружается, предсказания совпадают.

Рекомендации. Планируйте от 1000 размеченных предложений - рост со 100 до 1000 сократил ошибки втрое, и потолок ещё не достигнут. Больше всего примеров нужно для ORG, особенно там, где название компании похоже на имя человека или города. Отдельно проверяйте границы сущностей у кавычек, скобок и запятых.

Оговорка. Мы обучались на WikiANN - коротких энциклопедических фрагментах с автоматической разметкой, где часть сущностей пропущена. Ваши новости и соцсети устроены иначе, поэтому 0.80 на реальных данных не повторится. Снимите пилот на 200–300 своих предложений, чтобы получить честную отправную точку.

## Ресурсы

- [Hugging Face: Token Classification](https://huggingface.co/docs/transformers/tasks/token_classification)
- [Hugging Face Datasets](https://huggingface.co/docs/datasets/)
- [seqeval](https://github.com/chakki-works/seqeval)
- [WikiANN](https://huggingface.co/datasets/unimelb-nlp/wikiann)
- [bert-base-multilingual-cased](https://huggingface.co/bert-base-multilingual-cased)


## Чек-лист готовности второй части проекта

### Эксперименты

- [x] Настройки экспериментов собраны в одном месте, `SEED` зафиксирован.
- [x] Валидационный срез используется для выбора настроек, тестовая выборка — только для итоговой оценки.
- [x] Каждый запуск начинается с новой модели с корректными `num_labels`, `id2label` и `label2id`.
- [x] Метрики считаются через `seqeval` на уровне сущностей `PER`, `ORG` и `LOC`, позиции `-100` исключены.
- [x] Получен baseline без вызова `train()`.
- [x] На 100 примерах проверены два значения `learning_rate`, лучший вариант выбран по валидационному `F1`.
- [x] Модель на 1000 примеров обучена с выбранным `learning_rate` и теми же остальными настройками.
- [x] Обе обученные модели оценены на одной полной тестовой выборке.
- [x] Все три варианта модели представлены в одной сравнительной таблице.
- [x] Рассчитан абсолютный и относительный прирост качества после увеличения выборки.
- [x] Сформулированы выводы о роли дообучения и влиянии объёма обучающих данных.

### Анализ и итог

- [x] Показано минимум пять ошибок модели на увеличенной выборке.
- [x] Посчитаны частоты типов ошибок.
- [x] Найдены примеры, в которых каждая из двух обученных моделей превосходит другую.
- [x] Лучшая модель сохранена и повторно загружена, предсказания совпадают.
- [x] Выполнен инференс на 3–5 собственных предложениях с демонстрацией результатов.
- [x] Написаны выводы по каждому блоку и финальный вывод, в котором зафиксированы конкретные значения и ограничения.
- [x] Предоставлены рекомендации компании «Эхолот».
- [x] Описаны нюансы, связанные с отличиями текстов Википедии и «Эхолота».

### Формальные требования

- [x] Во всех случайных операциях используется единый `SEED`.
- [x] Все обязательные `TODO` заполнены, временные заглушки удалены.
- [x] Ядро перезапущено, тетрадка выполнена сверху вниз без ошибок.
- [x] Результаты вычислений, таблицы и визуализации сохранены в тетрадке.
- [x] Веса моделей не добавлены в репозиторий.